# 用免费 GPU 训练你自己的中文语音模型

这个笔记本把 [voice-chat-generation](https://github.com/6walnuts/voice-chat-generation) 采集的录音数据集，
在 Google Colab 的**免费 GPU** 上微调成你的 GPT-SoVITS 声音模型。

**你需要准备：**
1. 一个 Google 账号（用来跑 Colab）
2. 录音数据集 zip（「留声集」页面点「导出数据集」得到的 `voice-dataset-*.zip`，
   或仓库服务器版 `dataset/` 目录自己压缩的 zip——里面要有 `wavs/` 和 `gptsovits.list`）

**用法：** 菜单栏 `代码执行程序 → 全部运行`，跟着每个单元格的输出走。
整个过程约 30~60 分钟（安装 15 分钟 + 训练 20~40 分钟）。

> ⚠️ 先确认用的是 GPU：菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`。


## 第 1 步 · 确认分到了 GPU


In [ ]:
!nvidia-smi


## 第 2 步 · 安装 GPT-SoVITS（约 10~15 分钟，装完会自动下载预训练底模）


In [ ]:
%cd /content
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS
!pip install -q -r extra-req.txt --no-deps
!pip install -q -r requirements.txt
# GPT-SoVITS 锁定的旧版 opencc 在 Colab 新版 Python 上没有预编译包，源码编译会失败；
# 换带 wheel 的新版（API 兼容，中文文本处理需要）
!pip install -q --upgrade opencc

# 预训练底模（微调的起点），从 Hugging Face 官方仓库拉取
from huggingface_hub import snapshot_download
snapshot_download('lj1995/GPT-SoVITS', local_dir='GPT_SoVITS/pretrained_models')
print('✔ GPT-SoVITS 与预训练底模就绪')


## 第 3 步 · 上传你的数据集 zip

运行后会弹出上传框；zip 较大时建议先传到 Google Drive，再把路径填进 `DATASET_ZIP`。


In [ ]:
DATASET_ZIP = ''  # 可选：Google Drive 路径，如 /content/drive/MyDrive/voice-dataset.zip；留空则弹出上传框

import glob, os, wave
if DATASET_ZIP:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    from google.colab import files
    up = files.upload()
    DATASET_ZIP = '/content/' + list(up.keys())[0]

!rm -rf /content/dataset && mkdir -p /content/dataset
!unzip -qo "{DATASET_ZIP}" -d /content/dataset

# 自动定位（兼容 zip 里多套一层文件夹的情况）
hits = glob.glob('/content/dataset/**/gptsovits.list', recursive=True)
assert hits, 'zip 里没找到 gptsovits.list！请用「留声集」导出的 zip，或确认压缩包里包含它'
ROOT = os.path.dirname(hits[0])
wavs = glob.glob(os.path.join(ROOT, 'wavs', '*.wav'))
assert wavs, f'{ROOT}/wavs 里没有 wav 文件！'
total = 0.0
for q in wavs:
    with wave.open(q) as w:
        total += w.getnframes() / w.getframerate()
n_list = len(open(os.path.join(ROOT, 'gptsovits.list'), encoding='utf-8').read().splitlines())
print(f'✔ 数据集就绪：{len(wavs)} 条录音，共 {total/60:.1f} 分钟，标注 {n_list} 条')
print()
print('===== 待会儿在网页 1A 里要填的两个路径（复制粘贴用） =====')
print('文本标注文件      =', os.path.join(ROOT, 'gptsovits.list'))
print('训练集音频文件目录 =', os.path.join(ROOT, 'wavs'))


## 第 4 步 · 启动 GPT-SoVITS 网页

运行后在输出里找 `https://xxxx.gradio.live` 的链接并点开（保持本单元格一直运行，别停）。


In [ ]:
%cd /content/GPT-SoVITS
%env is_share=True
!python webui.py zh_CN


## 第 5 步 · 在打开的网页里点三步

**跳过「0-前置数据集获取工具」**（我们的数据已精确标注，不需要切割和语音识别）。

1. **1A-训练集格式化工具**：实验名随意（如 `myvoice`）；
   「文本标注文件」填 `/content/dataset/gptsovits.list`，
   「训练集音频文件目录」填 `/content/dataset/wavs`，点 **开启一键三连**，等它跑完。
2. **1B-微调训练**：先点 **开启 SoVITS 训练**（默认参数即可），完成后点 **开启 GPT 训练**。
3. **1C-推理**：刷新模型路径，选中刚训出的两个权重 → **开启 TTS 推理 WebUI**
   （会再给一个 gradio 链接）→ 上传一条你录得最满意的 3~10 秒录音当参考音频，
   参考文本从 `gptsovits.list` 里复制对应句子 → 输入任意文本 → **合成**，试听效果。

> 训练各需十几分钟；页面上进度不动时看本单元格的输出日志。


## 第 6 步 · 把训练好的模型下载回家

⚠️ Colab 会话关闭后所有文件都会消失——**满意后一定要跑这一格**把模型带走。
（先停止上一格的网页单元格，再运行这一格。）


In [ ]:
%cd /content/GPT-SoVITS
!zip -qr /content/my_voice_models.zip SoVITS_weights* GPT_weights*
from google.colab import files
files.download('/content/my_voice_models.zip')
print('✔ 模型已打包下载：以后在任何装了 GPT-SoVITS 的机器上放回同名目录即可直接推理，无需重训')


## 之后

- 本地合成/接 API：见仓库 [docs/training.md](https://github.com/6walnuts/voice-chat-generation/blob/main/docs/training.md) 第 6 步
- 模型和录音都是你的声音资产：自己保存好，不要交给别人，也不要用于冒充任何人
- 觉得不够像？回「留声集」把 A 批录满、补 5~10 分钟自由说话，重跑本笔记本
